# Relatorio de Manutencao (2025 ate hoje)

Este notebook le o ODBC em `02-Referencias/Meus Dados/base.csv` e gera um relatorio com todos os lancamentos ligados a manutencao no periodo de 2025 ate a data atual.

## Saida
- Arquivo Excel com 3 abas:
  - `Detalhado`: todos os lancamentos encontrados
  - `Resumo_Filial`: total por filial
  - `Resumo_Conta`: total por conta contabil


In [ ]:
from pathlib import Path
from datetime import date
import re
import pandas as pd
import numpy as np

# Parametros principais
DATA_INICIO = '2025-01-01'      # inicio fixo solicitado
DATA_FIM = date.today().isoformat()
CAMPO_DATA = 'lancamento'       # 'lancamento' ou 'ite_pagrec_vencimento'

# Critérios de manutencao
# 1) Termos em texto (descdc, observacao, documento)
TERMOS_MANUTENCAO = [
    'MANUTENCAO',
    'MANUTENÇÃO',
    'MANUT',
    'OFICINA',
    'MECANICA',
    'MECÂNICA',
    'BORRACHARIA',
    'PECA',
    'PEÇA'
]

# 2) Contas contabeis diretamente relacionadas (ajuste se necessario)
CONTAS_MANUTENCAO = [
    '7.1.2'  # MANUTENCAO DE VEICULOS/MAQUINAS
]

# Caminhos
REFS_DIR = Path('..') / '..' / '02-Referencias'
BASE_PATH = REFS_DIR / 'Meus Dados' / 'base.csv'
OUT_DIR = REFS_DIR

if not BASE_PATH.exists():
    raise FileNotFoundError(f'Arquivo nao encontrado: {BASE_PATH.resolve()}')

print('Parametros carregados.')
print(f'Periodo: {DATA_INICIO} ate {DATA_FIM}')
print(f'Campo de data: {CAMPO_DATA}')


In [ ]:
def read_csv_with_fallback(path, sep=';', dtype=str):
    encodings = ['utf-8', 'utf-8-sig', 'cp1252', 'latin1']
    ultimo_erro = None
    for enc in encodings:
        try:
            df = pd.read_csv(path, sep=sep, dtype=dtype, encoding=enc, low_memory=False)
            return df, enc
        except UnicodeDecodeError as e:
            ultimo_erro = e
    raise UnicodeDecodeError(
        getattr(ultimo_erro, 'encoding', 'unknown'),
        getattr(ultimo_erro, 'object', b''),
        getattr(ultimo_erro, 'start', 0),
        getattr(ultimo_erro, 'end', 1),
        f'Nao foi possivel ler com encodings {encodings}: {ultimo_erro}'
    )

def to_float_br(v):
    if pd.isna(v):
        return np.nan
    s = str(v).strip()
    if not s:
        return np.nan
    s = s.replace('.', '').replace(',', '.')
    try:
        return float(s)
    except ValueError:
        return np.nan

def parse_data_coluna(serie, campo_data):
    s = serie.fillna('').astype(str).str.strip()
    if campo_data == 'lancamento':
        # formato esperado: dd/mm/aaaa
        return pd.to_datetime(s, dayfirst=True, errors='coerce')
    # fallback padrao: aaaa-mm-dd
    return pd.to_datetime(s, errors='coerce')

def build_text_mask(df, termos):
    termos_limpos = [t.strip() for t in termos if str(t).strip()]
    if not termos_limpos:
        return pd.Series(False, index=df.index)
    padrao = '|'.join([re.escape(t) for t in termos_limpos])
    cols_texto = ['descdc', 'observacao', 'documento']
    mask = pd.Series(False, index=df.index)
    for c in cols_texto:
        if c in df.columns:
            mask = mask | df[c].fillna('').astype(str).str.contains(padrao, case=False, regex=True)
    return mask


In [ ]:
df, enc = read_csv_with_fallback(BASE_PATH)
print(f'Encoding detectado para base.csv: {enc}')

if CAMPO_DATA not in df.columns:
    raise KeyError(f"Campo de data '{CAMPO_DATA}' nao existe no base.csv")

dt = parse_data_coluna(df[CAMPO_DATA], CAMPO_DATA)
inicio = pd.to_datetime(DATA_INICIO)
fim = pd.to_datetime(DATA_FIM)
mask_periodo = dt.between(inicio, fim, inclusive='both')

mask_conta = df['codcdc'].fillna('').astype(str).isin(CONTAS_MANUTENCAO) if 'codcdc' in df.columns else pd.Series(False, index=df.index)
mask_texto = build_text_mask(df, TERMOS_MANUTENCAO)
mask_manutencao = mask_conta | mask_texto

rel = df.loc[mask_periodo & mask_manutencao].copy().reset_index(drop=True)

if rel.empty:
    raise ValueError('Nenhum lancamento de manutencao encontrado no periodo informado. Ajuste termos/contas ou campo de data.')

# Enriquecimento
rel['data_ref'] = parse_data_coluna(rel[CAMPO_DATA], CAMPO_DATA)
rel['ano'] = rel['data_ref'].dt.year
rel['mes'] = rel['data_ref'].dt.month
rel['valor_bruto_num'] = rel['valor_bruto'].apply(to_float_br) if 'valor_bruto' in rel.columns else np.nan
rel['valor_plano_num'] = rel['valor_plano'].apply(to_float_br) if 'valor_plano' in rel.columns else np.nan
rel['valor_centro_num'] = rel['valor_centro'].apply(to_float_br) if 'valor_centro' in rel.columns else np.nan

# Ordenacao para leitura
ordem = [c for c in ['data_ref', 'ano', 'mes', 'filial', 'codcen', 'descen', 'codcdc', 'descdc', 'documento', 'nome', 'valor_bruto', 'valor_plano', 'valor_centro', 'observacao', 'nota'] if c in rel.columns]
resto = [c for c in rel.columns if c not in ordem]
rel = rel[ordem + resto]

display(rel.head(10))
print(f'Total de lancamentos encontrados: {len(rel):,}'.replace(',', '.'))


In [ ]:
# Resumos
if 'valor_bruto_num' in rel.columns:
    resumo_filial = (
        rel.groupby('filial', dropna=False, as_index=False)['valor_bruto_num']
        .sum()
        .sort_values('valor_bruto_num', ascending=False)
    ) if 'filial' in rel.columns else pd.DataFrame()
else:
    resumo_filial = pd.DataFrame()

if 'valor_bruto_num' in rel.columns:
    resumo_conta = (
        rel.groupby(['codcdc', 'descdc'], dropna=False, as_index=False)['valor_bruto_num']
        .sum()
        .sort_values('valor_bruto_num', ascending=False)
    ) if {'codcdc', 'descdc'}.issubset(rel.columns) else pd.DataFrame()
else:
    resumo_conta = pd.DataFrame()

display(resumo_filial.head(20))
display(resumo_conta.head(20))


In [ ]:
nome_saida = f"RELATORIO_MANUTENCAO_{DATA_INICIO.replace('-', '')}_ate_{DATA_FIM.replace('-', '')}.xlsx"
saida = OUT_DIR / nome_saida

with pd.ExcelWriter(saida, engine='openpyxl') as writer:
    rel.to_excel(writer, index=False, sheet_name='Detalhado')
    resumo_filial.to_excel(writer, index=False, sheet_name='Resumo_Filial')
    resumo_conta.to_excel(writer, index=False, sheet_name='Resumo_Conta')

    # Cabeçalhos em negrito
    from openpyxl.styles import Font
    for nome_aba in ['Detalhado', 'Resumo_Filial', 'Resumo_Conta']:
        ws = writer.book[nome_aba]
        for cell in ws[1]:
            cell.font = Font(bold=True)

print(f'Relatorio salvo em: {saida.resolve()}')
print(f'Qtd. lancamentos: {len(rel):,}'.replace(',', '.'))
